In [ ]:
!pip install pyspark==3.5.1

In [ ]:
# Import SparkSession from PySpark
import pyspark

print(pyspark.__version__)

3.5.1


In [ ]:
# Import SparkSession from PySpark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Assignment 5") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Started Successfully")

Spark Started Successfully


In [ ]:
# Create sample employee dataset with null values and duplicate records
data = [
    (1, "Rishi", 22, "IT", 50000),
    (2, "Aman", 24, "HR", 45000),
    (3, "Priya", None, "Finance", 60000),
    (4, "Neha", 29, "IT", None),
    (5, "Karan", 31, None, 70000),
    (6, "Simran", 27, "HR", 48000),
    (7, "Rohit", None, "IT", 55000),
    (8, "Anjali", 25, "Finance", 62000),
    (2, "Aman", 24, "HR", 45000),   # Duplicate
    (9, "Vikas", 35, "IT", 80000)
]

columns = ["ID", "Name", "Age", "Department", "Salary"]

df = spark.createDataFrame(data, columns)

In [ ]:
df.show()

+---+------+----+----------+------+
| ID|  Name| Age|Department|Salary|
+---+------+----+----------+------+
|  1| Rishi|  22|        IT| 50000|
|  2|  Aman|  24|        HR| 45000|
|  3| Priya|NULL|   Finance| 60000|
|  4|  Neha|  29|        IT|  NULL|
|  5| Karan|  31|      NULL| 70000|
|  6|Simran|  27|        HR| 48000|
|  7| Rohit|NULL|        IT| 55000|
|  8|Anjali|  25|   Finance| 62000|
|  2|  Aman|  24|        HR| 45000|
|  9| Vikas|  35|        IT| 80000|
+---+------+----+----------+------+



In [ ]:
# Display the schema of the DataFrame
df.printSchema()

root
 |-- ID: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- Department: string (nullable = true)
 |-- Salary: long (nullable = true)



In [ ]:
# Count null values in each column
from pyspark.sql.functions import col, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+---+----+---+----------+--------------+-----+
| ID|Name|Age|Department|Monthly_Salary|Bonus|
+---+----+---+----------+--------------+-----+
|  0|   0|  0|         0|             0|    0|
+---+----+---+----------+--------------+-----+



In [ ]:
# Replace null values with default values
df = df.fillna({
    "Age": 0,
    "Department": "Unknown",
    "Salary": 0
})

df.show()

+---+------+---+----------+------+
| ID|  Name|Age|Department|Salary|
+---+------+---+----------+------+
|  1| Rishi| 22|        IT| 50000|
|  2|  Aman| 24|        HR| 45000|
|  3| Priya|  0|   Finance| 60000|
|  4|  Neha| 29|        IT|     0|
|  5| Karan| 31|   Unknown| 70000|
|  6|Simran| 27|        HR| 48000|
|  7| Rohit|  0|        IT| 55000|
|  8|Anjali| 25|   Finance| 62000|
|  2|  Aman| 24|        HR| 45000|
|  9| Vikas| 35|        IT| 80000|
+---+------+---+----------+------+



In [ ]:
# Remove duplicate rows from the dataset
print("Before:", df.count())

df = df.dropDuplicates()

print("After:", df.count())

Before: 10
After: 9


In [ ]:
# Filter
df.filter(df.Age > 25).show()

+---+------+---+----------+--------------+------+
| ID|  Name|Age|Department|Monthly_Salary| Bonus|
+---+------+---+----------+--------------+------+
|  5| Karan| 31|   Unknown|         70000|7000.0|
|  4|  Neha| 29|        IT|             0|   0.0|
|  9| Vikas| 35|        IT|         80000|8000.0|
|  6|Simran| 27|        HR|         48000|4800.0|
+---+------+---+----------+--------------+------+



In [ ]:
df.filter(df.Department == "IT").show()

+---+-----+---+----------+--------------+------+
| ID| Name|Age|Department|Monthly_Salary| Bonus|
+---+-----+---+----------+--------------+------+
|  4| Neha| 29|        IT|             0|   0.0|
|  1|Rishi| 22|        IT|         50000|5000.0|
|  9|Vikas| 35|        IT|         80000|8000.0|
|  7|Rohit|  0|        IT|         55000|5500.0|
+---+-----+---+----------+--------------+------+



In [ ]:
# Calculate average salary department-wise
from pyspark.sql.functions import avg

df.groupBy("Department") \
  .agg(avg("Monthly_Salary").alias("Average_Salary")) \
  .show()

+----------+--------------+
|Department|Average_Salary|
+----------+--------------+
|        HR|       46500.0|
|   Finance|       61000.0|
|   Unknown|       70000.0|
|        IT|       46250.0|
+----------+--------------+



In [ ]:
# Calculate total salary department-wise
from pyspark.sql.functions import sum

df.groupBy("Department") \
  .agg(sum("Monthly_Salary").alias("Total_Salary")) \
  .show()

+----------+------------+
|Department|Total_Salary|
+----------+------------+
|        HR|       93000|
|   Finance|      122000|
|   Unknown|       70000|
|        IT|      185000|
+----------+------------+



In [ ]:
# Count employees in each department
from pyspark.sql.functions import count

df.groupBy("Department") \
  .agg(count("*").alias("Employee_Count")) \
  .show()

+----------+--------------+
|Department|Employee_Count|
+----------+--------------+
|        HR|             2|
|   Finance|             2|
|   Unknown|             1|
|        IT|             4|
+----------+--------------+



In [ ]:
# Create a new derived column 'Bonus' (10% of Salary)
from pyspark.sql.functions import col

df = df.withColumn("Bonus", col("Monthly_Salary") * 0.10)

df.show()

+---+------+---+----------+--------------+------+
| ID|  Name|Age|Department|Monthly_Salary| Bonus|
+---+------+---+----------+--------------+------+
|  5| Karan| 31|   Unknown|         70000|7000.0|
|  4|  Neha| 29|        IT|             0|   0.0|
|  2|  Aman| 24|        HR|         45000|4500.0|
|  3| Priya|  0|   Finance|         60000|6000.0|
|  1| Rishi| 22|        IT|         50000|5000.0|
|  9| Vikas| 35|        IT|         80000|8000.0|
|  7| Rohit|  0|        IT|         55000|5500.0|
|  8|Anjali| 25|   Finance|         62000|6200.0|
|  6|Simran| 27|        HR|         48000|4800.0|
+---+------+---+----------+--------------+------+



In [ ]:
df.printSchema()

root
 |-- ID: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: long (nullable = false)
 |-- Department: string (nullable = false)
 |-- Salary: long (nullable = false)
 |-- Bonus: double (nullable = false)



In [ ]:
df = df.withColumnRenamed("Salary", "Monthly_Salary")

df.show()

+---+------+---+----------+--------------+------+
| ID|  Name|Age|Department|Monthly_Salary| Bonus|
+---+------+---+----------+--------------+------+
|  5| Karan| 31|   Unknown|         70000|7000.0|
|  4|  Neha| 29|        IT|             0|   0.0|
|  2|  Aman| 24|        HR|         45000|4500.0|
|  3| Priya|  0|   Finance|         60000|6000.0|
|  1| Rishi| 22|        IT|         50000|5000.0|
|  9| Vikas| 35|        IT|         80000|8000.0|
|  7| Rohit|  0|        IT|         55000|5500.0|
|  8|Anjali| 25|   Finance|         62000|6200.0|
|  6|Simran| 27|        HR|         48000|4800.0|
+---+------+---+----------+--------------+------+



In [ ]:
# Sort employees by Monthly Salary in descending order
from pyspark.sql.functions import col

df.orderBy(col("Monthly_Salary").desc()).show()

+---+------+---+----------+--------------+------+
| ID|  Name|Age|Department|Monthly_Salary| Bonus|
+---+------+---+----------+--------------+------+
|  9| Vikas| 35|        IT|         80000|8000.0|
|  5| Karan| 31|   Unknown|         70000|7000.0|
|  8|Anjali| 25|   Finance|         62000|6200.0|
|  3| Priya|  0|   Finance|         60000|6000.0|
|  7| Rohit|  0|        IT|         55000|5500.0|
|  1| Rishi| 22|        IT|         50000|5000.0|
|  6|Simran| 27|        HR|         48000|4800.0|
|  2|  Aman| 24|        HR|         45000|4500.0|
|  4|  Neha| 29|        IT|             0|   0.0|
+---+------+---+----------+--------------+------+



In [ ]:
df.filter(
    (col("Department") == "IT") &
    (col("Monthly_Salary") > 50000)
).show()

+---+-----+---+----------+--------------+------+
| ID| Name|Age|Department|Monthly_Salary| Bonus|
+---+-----+---+----------+--------------+------+
|  9|Vikas| 35|        IT|         80000|8000.0|
|  7|Rohit|  0|        IT|         55000|5500.0|
+---+-----+---+----------+--------------+------+



In [ ]:
# Final Schema
df.printSchema()

root
 |-- ID: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = false)
 |-- Department: string (nullable = false)
 |-- Monthly_Salary: integer (nullable = false)
 |-- Bonus: double (nullable = false)



In [ ]:
# Datatype Casting
from pyspark.sql.functions import col

df = df.withColumn("Age", col("Age").cast("integer"))
df = df.withColumn("Monthly_Salary", col("Monthly_Salary").cast("integer"))

df.printSchema()

root
 |-- ID: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = false)
 |-- Department: string (nullable = false)
 |-- Monthly_Salary: integer (nullable = false)
 |-- Bonus: double (nullable = false)



In [46]:
df.coalesce(1).write.mode("overwrite").option("header", True).csv("cleaned_employee_data")